# Notebook 05: Semantic Classification Model Training

**Phase:** Semantic Classification (Thesis Plan Deliverable 1)

**Purpose:** Train a multi-label classifier for semantic ingredient categorization using the labeled data from Notebook 04.

## Overview

Trains a MobileBERT-based multi-label classifier for semantic ingredient categories.
Reuses the same architecture and patterns from the allergen detection model (Notebook 07)
but with a different label space and task.

## Workflow

1. Load semantic labeling data from Notebook 04
2. Prepare multi-label dataset (ingredient text → semantic category vector)
3. Stratified train/val/test split
4. Train MobileBERT with weighted BCE loss
5. Evaluate and optimize per-class thresholds
6. Save model and evaluation metrics

In [1]:
import os
import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score

from utils.data_utils import load_labeled_data, create_stratified_splits, get_data_directories, save_metadata
from utils.semantic_utils import get_semantic_category_list, get_category_groups
from utils.model_utils import load_model_and_tokenizer, compute_class_weights, find_best_thresholds, predict_ml
from utils.evaluation import print_classification_report, error_analysis

# Set seeds for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports completed.")

Imports completed.


## 1. Load Semantic Label Data

In [2]:
dirs = get_data_directories()
categories = get_semantic_category_list()
print(f"Semantic categories ({len(categories)}): {categories}")

# Load the ingredient-level training data
training_path = os.path.join(dirs['final'], 'semantic_training_data.csv')
data = pd.read_csv(training_path)

# Convert label vectors from strings back to lists
import ast
data['label_vector'] = data['label_vector'].apply(ast.literal_eval)

# Pad label vectors if the category list grew after the data was saved
n_loaded = len(data['label_vector'].iloc[0])
n_expected = len(categories)
if n_loaded < n_expected:
    pad_width = n_expected - n_loaded
    data['label_vector'] = data['label_vector'].apply(
        lambda v: v + [0] * pad_width
    )
    print(f"Padded label vectors: {n_loaded} → {n_expected} dimensions "
          f"(+{pad_width} new categories: {categories[n_loaded:]})")
elif n_loaded > n_expected:
    print(f"WARNING: Loaded {n_loaded}-dim vectors but expected {n_expected}. "
          "Truncating — data may be misaligned with current category list.")
    data['label_vector'] = data['label_vector'].apply(
        lambda v: v[:n_expected]
    )

print(f"\nTraining data shape: {data.shape}")
print(f"Products represented: {data['product_code'].nunique()}")
print(f"Unique ingredients: {data['ingredient'].nunique()}")
print(f"\nFirst 5 rows:")
data[['ingredient', 'semantic_labels']].head()

Semantic categories (40): ['food_additive', 'preservative', 'flavor_enhancer', 'sweetener', 'emulsifier', 'stabilizer', 'thickener', 'antioxidant', 'acidulant', 'colorant', 'fat_source', 'oil_source', 'protein_source', 'carbohydrate_source', 'animal_derived', 'plant_derived', 'milk_derivative', 'egg_derivative', 'soy_derivative', 'wheat_derivative', 'sodium_compound', 'sugar', 'added_sugar', 'fiber', 'vitamin', 'mineral', 'salt', 'yeast', 'leavening_agent', 'gelling_agent', 'humectant', 'flavoring', 'spice', 'fruit_derived', 'vegetable_derived', 'fermented', 'cured', 'enzyme', 'culture', 'salt_substitute']

Training data shape: (18621, 7)
Products represented: 1057
Unique ingredients: 4655

First 5 rows:


,ingredient,semantic_labels
0,wheat flour,"['wheat_derivative', 'plant_derived', 'carbohy..."
1,vitamin b2,['vitamin']
2,vitamin b3,['vitamin']
3,vitamin a,['vitamin']
4,purified water,['mineral']


## 2. Prepare Training Data

Split into train/val/test using stratified multi-label splitting.

In [3]:
# Prepare feature matrix and labels
X = data['ingredient'].values
y = np.array(data['label_vector'].tolist())

print(f"Feature matrix: {X.shape}")
print(f"Label matrix: {y.shape}")
print(f"Positive label ratio: {y.sum() / y.size:.2%}")

# Stratified split
train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = create_stratified_splits(
    X, y,
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
    random_state=SEED
)

num_unique_train = len(set(tuple(l) for l in train_labels))
print(f"\nSplit sizes:")
print(f"  Train: {len(train_texts)} ({num_unique_train} unique label combos)")
print(f"  Val:   {len(val_texts)}")
print(f"  Test:  {len(test_texts)}")

Feature matrix: (18621,)
Label matrix: (18621, 40)
Positive label ratio: 5.73%

Split sizes:
  Train: 14896 (411 unique label combos)
  Val:   1862
  Test:  1863


## 3. Compute Class Weights

Handle class imbalance using inverse frequency weighting (same approach as allergen model).

In [4]:
# Compute weights for each semantic category
pos_counts = np.array(train_labels).sum(axis=0)
class_weights = compute_class_weights(torch.tensor(train_labels, dtype=torch.float32))

print("Class weights (inverse frequency):")
for i, (cat, weight) in enumerate(zip(categories, class_weights.numpy())):
    w_str = f"{weight:.2f}"
    counts_str = f"{int(pos_counts[i])}"
    flag = " ← rare" if weight > 3 else ""
    print(f"  {cat:30s}  weight={w_str:>6s}  positive_samples={counts_str}{flag}")

Class weights (inverse frequency):
  food_additive                   weight=  0.33  positive_samples=4722
  preservative                    weight=  0.94  positive_samples=584
  flavor_enhancer                 weight=  0.90  positive_samples=653
  sweetener                       weight=  0.67  positive_samples=1470
  emulsifier                      weight=  0.89  positive_samples=676
  stabilizer                      weight=  0.90  positive_samples=666
  thickener                       weight=  0.90  positive_samples=669
  antioxidant                     weight=  0.89  positive_samples=693
  acidulant                       weight=  0.88  positive_samples=718
  colorant                        weight=  0.87  positive_samples=734
  fat_source                      weight=  0.66  positive_samples=1525
  oil_source                      weight=  0.79  positive_samples=982
  protein_source                  weight=  0.73  positive_samples=1178
  carbohydrate_source             weight=  0.59  po

## 4. Load Model

Initialize a fresh MobileBERT for semantic classification (separate from the allergen model).
The num_labels is set to the number of semantic categories.

In [5]:
MODEL_NAME = "google/mobilebert-uncased"
NUM_CATEGORIES = len(categories)
MAX_LENGTH = 32  # Individual ingredients are short
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 2e-5

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CATEGORIES,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True,  # new classification head
)

# Move to available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded on {device}")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Classifier: nn.Linear({model.config.hidden_size}, {NUM_CATEGORIES})")
print(f"  Epoch 1 loss will be high (~1M+), but this is cosmetic — gradient clipping handles it")

Loading google/mobilebert-uncased...


Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded on cuda
  Layers: 24
  Parameters: 24,602,408
  Classifier: nn.Linear(512, 40)
  Epoch 1 loss will be high (~1M+), but this is cosmetic — gradient clipping handles it


## 5. Training Setup

Uses the same weighted BCE loss and training loop as the allergen model.
Early stopping with patience=3 monitors validation macro F1.

In [6]:
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Tokenize all data
def tokenize_batch(texts):
    return tokenizer(
        texts.tolist() if hasattr(texts, 'tolist') else list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )

train_encodings = tokenize_batch(train_texts)
val_encodings = tokenize_batch(val_texts)

# Create DataLoaders
train_dataset = TensorDataset(
    train_encodings['input_ids'],
    train_encodings['attention_mask'],
    torch.tensor(train_labels, dtype=torch.float32),
)
val_dataset = TensorDataset(
    val_encodings['input_ids'],
    val_encodings['attention_mask'],
    torch.tensor(val_labels, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS

# === FIX: Ensure at least 200 warmup steps ===
# Linear warmup starts learning rate from 0, preventing the optimizer from
# amplifying the initial extreme logits from the randomly-initialized head.
# 10% of total steps can be <200 for small datasets; cap at 200 minimum.
num_warmup_steps = max(200, int(0.1 * total_steps))
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps
)

print(f"Training setup:")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Total steps: {total_steps}")
print(f"  Warmup steps: {num_warmup_steps}")

Training setup:
  Train samples: 14896
  Validation samples: 1862
  Batch size: 16
  Max length: 32
  Learning rate: 2e-05
  Epochs: 10
  Total steps: 9310
  Warmup steps: 931


## 6. Training Loop

Weighted BCE loss, early stopping by macro F1 on validation set.
Training code follows the same pattern as Notebook 07 (allergen training).

In [7]:
# === TRAINING LOOP ===

from sklearn.metrics import f1_score

loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights.to(device))

best_macro_f1 = 0.0
best_epoch = 0
patience_counter = 0

# Save path
semantic_model_path = "../models/mobilebert_semantic_final/"
os.makedirs(semantic_model_path, exist_ok=True)

for epoch in range(EPOCHS):
    # Training
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = [x.to(device) for x in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    
    # Validation
    model.eval()
    val_preds, val_probs = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = [x.to(device) for x in batch]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits)
            val_probs.append(probs.cpu().numpy())
    
    val_probs = np.concatenate(val_probs)
    val_preds = (val_probs >= 0.5).astype(int)
    macro_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
    avg_loss = total_loss / len(train_loader)
    
    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={avg_loss:.4f}  val_macro_f1={macro_f1:.4f}")
    
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_epoch = epoch
        patience_counter = 0
        # Save best model
        model.save_pretrained(semantic_model_path)
        tokenizer.save_pretrained(semantic_model_path)
        print(f"  → New best model saved (macro_f1={macro_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= 3:
            print(f"  Early stopping at epoch {epoch+1}")
            break

Epoch  1/10  loss=375971.4457  val_macro_f1=0.7155
  → New best model saved (macro_f1=0.7155)
Epoch  2/10  loss=0.4042  val_macro_f1=0.9251
  → New best model saved (macro_f1=0.9251)
Epoch  3/10  loss=0.0329  val_macro_f1=0.9605
  → New best model saved (macro_f1=0.9605)
Epoch  4/10  loss=0.5285  val_macro_f1=0.9663
  → New best model saved (macro_f1=0.9663)
Epoch  5/10  loss=0.1782  val_macro_f1=0.9678
  → New best model saved (macro_f1=0.9678)
Epoch  6/10  loss=0.2863  val_macro_f1=0.9766
  → New best model saved (macro_f1=0.9766)
Epoch  7/10  loss=0.0079  val_macro_f1=0.9784
  → New best model saved (macro_f1=0.9784)
Epoch  8/10  loss=0.0054  val_macro_f1=0.9772
Epoch  9/10  loss=0.0144  val_macro_f1=0.9783
Epoch 10/10  loss=0.0631  val_macro_f1=0.9785
  → New best model saved (macro_f1=0.9785)


## 7. Threshold Optimization

Find optimal per-class probability thresholds (same approach as allergen model).

In [8]:
# Load best model and optimize thresholds
model_path = "../models/mobilebert_semantic_final/"
if os.path.exists(model_path) and any(os.scandir(model_path)):
    model, tokenizer, device = load_model_and_tokenizer(model_path)
    _, val_probs = predict_ml(val_texts, model, tokenizer, device,
                              thresholds=np.array([0.5] * NUM_CATEGORIES),
                              max_length=MAX_LENGTH)
    best_thresholds = find_best_thresholds(val_probs, np.array(val_labels))
    print("Optimized thresholds:")
    for i, cat in enumerate(categories):
        print(f"  {cat:30s}  threshold={best_thresholds[i]:.2f}")
else:
    print(f"Model not found at {model_path}. Train the model first in the cell above.")

Optimized thresholds:
  food_additive                   threshold=0.17
  preservative                    threshold=0.09
  flavor_enhancer                 threshold=0.58
  sweetener                       threshold=0.74
  emulsifier                      threshold=0.67
  stabilizer                      threshold=0.37
  thickener                       threshold=0.65
  antioxidant                     threshold=0.50
  acidulant                       threshold=0.30
  colorant                        threshold=0.93
  fat_source                      threshold=0.34
  oil_source                      threshold=0.25
  protein_source                  threshold=0.90
  carbohydrate_source             threshold=0.10
  animal_derived                  threshold=0.16
  plant_derived                   threshold=0.67
  milk_derivative                 threshold=0.27
  egg_derivative                  threshold=0.15
  soy_derivative                  threshold=0.21
  wheat_derivative                threshold=0.0

## 8. Evaluate on Test Set

In [9]:
model_path = "../models/mobilebert_semantic_final/"
if os.path.exists(model_path) and any(os.scandir(model_path)):
    test_preds, _ = predict_ml(test_texts, model, tokenizer, device, thresholds=best_thresholds, max_length=MAX_LENGTH)
    print_classification_report(test_labels, test_preds, categories, prefix="Semantic Classifier")
else:
    print(f"Model not found at {model_path}. Train and optimize thresholds first.")

=== Semantic Classifier ===
                     precision    recall  f1-score   support

      food_additive     0.9668    0.9848    0.9757       591
       preservative     0.9221    0.9726    0.9467        73
    flavor_enhancer     0.9643    0.9878    0.9759        82
          sweetener     0.9944    0.9728    0.9835       184
         emulsifier     0.9770    1.0000    0.9884        85
         stabilizer     0.9878    0.9759    0.9818        83
          thickener     1.0000    0.9643    0.9818        84
        antioxidant     0.9773    0.9885    0.9829        87
          acidulant     0.9667    0.9667    0.9667        90
           colorant     0.9889    0.9674    0.9780        92
         fat_source     0.9541    0.9791    0.9664       191
         oil_source     0.9758    0.9837    0.9798       123
     protein_source     0.9716    0.9257    0.9481       148
carbohydrate_source     0.9758    0.9918    0.9837       244
     animal_derived     0.9624    0.9890    0.9755      

## 9. Summary

In [10]:
print("=" * 50)
print("SEMANTIC CLASSIFICATION MODEL TRAINING")
print("=" * 50)
print(f"  Model: MobileBERT")
print(f"  Task: Multi-label semantic classification ({NUM_CATEGORIES} categories)")
print(f"  Training samples: {len(train_texts)}")
print(f"  Validation samples: {len(val_texts)}")
print(f"  Test samples: {len(test_texts)}")
print(f"  Max seq length: {MAX_LENGTH}")
print()
print("📋 Next step: Run 06_allergen_labeling.ipynb to continue allergen pipeline")

SEMANTIC CLASSIFICATION MODEL TRAINING
  Model: MobileBERT
  Task: Multi-label semantic classification (40 categories)
  Training samples: 14896
  Validation samples: 1862
  Test samples: 1863
  Max seq length: 32

📋 Next step: Run 06_allergen_labeling.ipynb to continue allergen pipeline
